# 19 文档边界 mask 如何防止 packed sequence 信息泄漏？

## 面试回答主线

packed sequence 中 causal mask 只能阻止看未来，不能阻止后一个文档读取前一个文档。文档边界 mask 必须将 query/key 的 segment id 纳入可见性条件；训练 loss 看起来下降也不能证明没有泄漏。面试时最好写出 mask 条件并检查跨文档 attention mass 是否为零。实验用一个含两段客服对话的 packed block 手写 scaled dot-product attention，先只用 causal mask，再叠加 segment mask，直接观察第二段首 token 对第一段的注意力质量。

**核心公式：** $A_{ij}=-\infty$ 若 $j>i$ 或 $s_i\ne s_j$，否则 $A_{ij}=q_i k_j/\sqrt d$；softmax 后跨文档 attention 概率应为零。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
hidden = torch.tensor([[1.0, 0.2], [0.8, 0.1], [0.1, 1.0], [0.2, 0.9]])  # 构造两段各两个 token 的隐藏表示。
segments = [0, 0, 1, 1]  # 标注前两 token 属于文档 A，后两 token 属于文档 B。
scores = hidden @ hidden.T / math.sqrt(2.0)  # 手写 Q=K=hidden 的缩放点积 attention 分数。
causal_mask = torch.tensor([[0.0 if column <= row else -1e9 for column in range(4)] for row in range(4)])  # 构造只限制未来的 causal mask。
causal_attention = torch.softmax(scores + causal_mask, dim=-1)  # 计算只用 causal mask 的 attention。
baseline_metric = float(causal_attention[2, :2].sum())  # 读取文档 B 首 token 对文档 A 的注意力质量。
print(f'仅 causal：B 首 token 对 A 的 attention mass={baseline_metric:.4f}，完整行={causal_attention[2].tolist()}')  # 暴露跨文档泄漏。


仅 causal：B 首 token 对 A 的 attention mass=0.5373，完整行=[0.28005358576774597, 0.2572705149650574, 0.46267586946487427, 0.0]


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
boundary_mask = torch.tensor([[0.0 if column <= row and segments[column] == segments[row] else -1e9 for column in range(4)] for row in range(4)])  # 同时编码因果和文档边界。
boundary_attention = torch.softmax(scores + boundary_mask, dim=-1)  # 计算带边界 mask 的 attention。
core_metric = float(boundary_attention[2, :2].sum())  # 再次读取跨文档 attention mass。
print(f'边界 mask：B 首 token 对 A 的 attention mass={core_metric:.4f}，完整行={boundary_attention[2].tolist()}')  # 输出修复后的 attention 行。


边界 mask：B 首 token 对 A 的 attention mass=0.0000，完整行=[0.0, 0.0, 1.0, 0.0]


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.537324
核心机制     | 指标=0.000000


## 结果解读

基线和核心输出只在本受控案例中比较。生产上要让 data collator、attention kernel、loss mask 和 position id 使用同一段边界；任何一处遗漏都可能重新打开泄漏通道。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
loss_mask_only = torch.tensor([1, 1, 1, 1])  # 构造错误的“只在 loss 端处理”掩码。
failure_metric = float(causal_attention[2, :2].sum() * loss_mask_only[2])  # 证明 loss mask 不会改变 attention 泄漏。
fix_metric = float(boundary_attention[2, :2].sum())  # 使用真正的 attention 边界 mask 作为修复。
print(f'失败：仅 loss mask 后跨文档 mass={failure_metric:.4f}；修复：attention 边界 mask 后 mass={fix_metric:.4f}')  # 区分两种 mask 的职责。


失败：仅 loss mask 后跨文档 mass=0.5373；修复：attention 边界 mask 后 mass=0.0000


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产上要让 data collator、attention kernel、loss mask 和 position id 使用同一段边界；任何一处遗漏都可能重新打开泄漏通道。

**常见坑：** 只在 loss 上 mask 跨文档 token，却没有在 attention 上 mask；模型仍能利用前一段上下文作弊。

**延伸追问：** prefix-LM、FIM 或跨文档检索拼接时，哪些 segment 允许互相可见？如何为 mask 编写单元测试和可视化？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert baseline_metric > 0.1  # 验证 causal-only 确实产生跨文档泄漏。
assert core_metric < 1e-6  # 验证边界 mask 将跨文档概率压到零。
assert failure_metric > fix_metric  # 验证 loss mask 不能替代 attention mask。
assert torch.isfinite(boundary_attention).all()  # 验证掩码 softmax 保持数值有限。
